In [7]:
import pandas as pd

df = pd.read_csv('datasets/company_data.csv')

print(df.shape)
df.head()

(10000, 9)


,Name,Rating,Reviews,Type,Headquater,Salaries,Interviews,Jobs,Benefits
0,TCS,3.2,(1.2L),IT Services & Consulting,Bengaluru,10.4L,11.4k,4.7k,11.1k
1,Accenture,3.7,(78k),IT Services & Consulting,Bengaluru,7.3L,9.6k,14.3k,7.1k
2,Wipro,3.6,(68.5k),IT Services & Consulting,Hyderabad,4.9L,7k,7,5k
3,Cognizant,3.7,(64.7k),IT Services & Consulting,Hyderabad,6L,6.6k,734,5.7k
4,Capgemini,3.6,(56.9k),IT Services & Consulting,Bengaluru,4.9L,5.7k,2k,3.9k


In [8]:
# Layer 2: Header Standardization
df.columns = df.columns.str.lower()
df.columns = df.columns.str.replace('headquater', 'headquarters')

# Check the memory bloated by these 'k' and 'L' strings
df.info(memory_usage='deep')

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   name          10000 non-null  str    
 1   rating        9984 non-null   float64
 2   reviews       9984 non-null   str    
 3   type          10000 non-null  str    
 4   headquarters  9698 non-null   str    
 5   salaries      10000 non-null  str    
 6   interviews    10000 non-null  str    
 7   jobs          10000 non-null  str    
 8   benefits      10000 non-null  str    
dtypes: float64(1), str(8)
memory usage: 1.2 MB


In [9]:
df.nunique()

name            9451
rating            38
reviews          947
type             318
headquarters     251
salaries        1219
interviews       473
jobs             308
benefits         468
dtype: int64

In [10]:
import numpy as np

# ==============================================================================
# Layer 3: Memory Optimization (Categorical Downcast & PyArrow Enforcement)
# ==============================================================================

# 1. Low Cardinality: Downcast to category for memory compression
df['type'] = df['type'].astype('category')
df['headquarters'] = df['headquarters'].astype('category')

# 2. High Cardinality: Enforce PyArrow to prevent string fragmentation
df['name'] = df['name'].astype('string[pyarrow]')

# ==============================================================================
# Layer 3 & 4: The Custom Multiplier Engine & Memory Optimizer
# ==============================================================================

target_cols = ['reviews', 'salaries', 'interviews', 'jobs', 'benefits']

# 2. NullHunter Custom Intelligence Function
def convert_metrics(val):
    # Agar pehle se hi NaN (missing) hai, toh safe rakho
    if pd.isna(val):
        return np.nan
    
    # String banakar spaces aur brackets () hatao
    val = str(val).replace('(', '').replace(')', '').strip()
    
    # Default multiplier 1 hai
    multiplier = 1
    
    # Agar aakhri character 'k' hai
    if val.endswith('k') or val.endswith('K'):
        multiplier = 1000
        val = val[:-1] # '11.4k' me se 'k' hata kar '11.4' rakho
        
    # Agar aakhri character 'L' hai
    elif val.endswith('L') or val.endswith('l'):
        multiplier = 100000
        val = val[:-1] # '10.4L' me se 'L' hata kar '10.4' rakho
        
    # Safe Coercion: Math apply karo
    try:
        return float(val) * multiplier
    except ValueError:
        return np.nan # Agar koi ajeeb text fasa ho toh crash na ho

# 3. Execution: Apply the engine to target columns and cast directly to float32
for col in target_cols:
    df[col] = df[col].apply(convert_metrics).astype('float32')

# Verify the transformation!
df.head()

,name,rating,reviews,type,headquarters,salaries,interviews,jobs,benefits
0,TCS,3.2,120000.0,IT Services & Consulting,Bengaluru,1040000.0,11400.0,4700.0,11100.0
1,Accenture,3.7,78000.0,IT Services & Consulting,Bengaluru,730000.0,9600.0,14300.0,7100.0
2,Wipro,3.6,68500.0,IT Services & Consulting,Hyderabad,490000.0,7000.0,7.0,5000.0
3,Cognizant,3.7,64700.0,IT Services & Consulting,Hyderabad,600000.0,6600.0,734.0,5700.0
4,Capgemini,3.6,56900.0,IT Services & Consulting,Bengaluru,490000.0,5700.0,2000.0,3900.0


In [11]:
# Check the final optimized memory footprint!
df.info(memory_usage='deep')

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype   
---  ------        --------------  -----   
 0   name          10000 non-null  string  
 1   rating        9984 non-null   float64 
 2   reviews       9984 non-null   float32 
 3   type          10000 non-null  category
 4   headquarters  9698 non-null   category
 5   salaries      9983 non-null   float32 
 6   interviews    9962 non-null   float32 
 7   jobs          6415 non-null   float32 
 8   benefits      9954 non-null   float32 
dtypes: category(2), float32(5), float64(1), string(1)
memory usage: 573.3 KB


In [12]:
df.isnull().sum()

name               0
rating            16
reviews           16
type               0
headquarters     302
salaries          17
interviews        38
jobs            3585
benefits          46
dtype: int64

In [13]:
from sklearn.impute import KNNImputer

# ==============================================================================
# Layer 4: Advanced Predictive Imputation (KNN Engine)
# ==============================================================================

# 1. Select numeric features that define a company's "Scale"
# We exclude 'jobs' for now because it has massive missing data (3585 NaNs)
knn_features = ['rating', 'reviews', 'salaries', 'interviews', 'benefits']

# 2. Initialize the KNN Imputer
# weights='distance' means closer companies have a stronger influence on the predicted rating
knn_imputer = KNNImputer(n_neighbors=5, weights='distance')

# 3. Execute the imputation safely on the selected columns
df[knn_features] = knn_imputer.fit_transform(df[knn_features])

# Verify if the 16 ratings and other small NaNs are mathematically solved
df.isnull().sum()

name               0
rating             0
reviews            0
type               0
headquarters     302
salaries           0
interviews         0
jobs            3585
benefits           0
dtype: int64

In [14]:
# ==============================================================================
# Layer 4.2: Business Logic & Categorical Imputation
# ==============================================================================

# 1. Headquarters (Categorical): The 'Unknown' Tag Strategy
# Add 'Unknown' to the category dictionary first, then fill
df['headquarters'] = df['headquarters'].cat.add_categories('Unknown')
df['headquarters'] = df['headquarters'].fillna('Unknown')

# 2. Jobs (Numeric/Logical): Missing job postings mean 0 active vacancies
df['jobs'] = df['jobs'].fillna(0.0)

# Final Verification: The Matrix should now be 100% pure
df.isnull().sum()

name            0
rating          0
reviews         0
type            0
headquarters    0
salaries        0
interviews      0
jobs            0
benefits        0
dtype: int64